In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
PROC = Path("../data/processed")

inkar_csv = RAW / "inkar_2025" / "inkar_2025.csv"

# First, read only 5 rows to confirm the file path and structure.
test = pd.read_csv(
    inkar_csv,
    sep=";",
    decimal=",",
    encoding="utf-8",
    dtype={"Kennziffer": str},
    nrows=5
)

print("File exists:", inkar_csv.exists())
print("Sample shape:", test.shape)
print("Columns:")
print(list(test.columns))

test


File exists: True
Sample shape: (5, 9)
Columns:
['Bereich', 'ID', 'Kuerzel', 'Indikator', 'Raumbezug', 'Kennziffer', 'Name', 'Zeitbezug', 'Wert']


,Bereich,ID,Kuerzel,Indikator,Raumbezug,Kennziffer,Name,Zeitbezug,Wert
0,EU,18668,q_alo,Arbeitslosenquote,EU27,EU27,Europäische Union - 27 Länder (ab 2020),2021,7.0
1,EU,18668,q_alo,Arbeitslosenquote,NUTS0,AT,Österreich,2021,6.2
2,EU,18668,q_alo,Arbeitslosenquote,NUTS0,BE,Belgien,2021,6.3
3,EU,18668,q_alo,Arbeitslosenquote,NUTS0,BG,Bulgarien,2021,5.3
4,EU,18668,q_alo,Arbeitslosenquote,NUTS0,CY,Zypern,2021,7.5


In [2]:
# Cell 2 — search indicator names at Gemeinde level

indicator_names = set()

for chunk in pd.read_csv(
    inkar_csv,
    sep=";",
    decimal=",",
    encoding="utf-8",
    dtype={"Kennziffer": str},
    usecols=["Raumbezug", "Indikator"],
    chunksize=500_000
):
    gem_chunk = chunk[chunk["Raumbezug"] == "Gemeinden"]
    indicator_names.update(gem_chunk["Indikator"].dropna().unique())

indicator_names = sorted(indicator_names)

print("Number of Gemeinde-level indicators:", len(indicator_names))

keywords = ["Arbeitslos", "SGB", "65", "Bevölkerung"]

for kw in keywords:
    print(f"\n--- Indicators containing '{kw}' ---")
    matches = [name for name in indicator_names if kw.lower() in name.lower()]
    for m in matches:
        print(m)

Number of Gemeinde-level indicators: 332

--- Indicators containing 'Arbeitslos' ---
Anteil jüngere Arbeitslose
Anteil männliche jüngere Arbeitslose
Anteil männliche ältere Arbeitslose
Anteil weibliche jüngere Arbeitslose
Anteil weibliche ältere Arbeitslose
Anteil ältere Arbeitslose
Arbeitslose
Arbeitslose Frauen
Arbeitslose Männer
Arbeitslose mit Anforderungsniveau Experte
Arbeitslose mit Anforderungsniveau Fachkraft
Arbeitslose mit Anforderungsniveau Helfer
Arbeitslose mit Anforderungsniveau Spezialist
Arbeitslose ohne Ausbildung
Arbeitslosigkeit
Arbeitslosigkeit - Allgemein
Arbeitslosigkeit - Altersgruppen
Arbeitslosigkeit - Struktur
Ausländische Arbeitslose
Ausländische männliche Arbeitslose
Ausländische weibliche Arbeitslose
Jüngere Arbeitslose
Langzeitarbeitslose
Männliche Langzeitarbeitslose
Männliche jüngere Arbeitslose
Männliche ältere Arbeitslose
Weibliche Langzeitarbeitslose
Weibliche jüngere Arbeitslose
Weibliche ältere Arbeitslose
Ältere Arbeitslose

--- Indicators contain

In [3]:
# Cell 3 — focused indicator search

def print_matches(keyword):
    matches = [name for name in indicator_names if keyword.lower() in name.lower()]
    print(f"\n--- {len(matches)} indicators containing '{keyword}' ---")
    for i, name in enumerate(matches, start=1):
        print(f"{i:02d}. {name}")

for keyword in ["Arbeitslos", "SGB", "65", "Bevölkerungsentwicklung"]:
    print_matches(keyword)


--- 30 indicators containing 'Arbeitslos' ---
01. Anteil jüngere Arbeitslose
02. Anteil männliche jüngere Arbeitslose
03. Anteil männliche ältere Arbeitslose
04. Anteil weibliche jüngere Arbeitslose
05. Anteil weibliche ältere Arbeitslose
06. Anteil ältere Arbeitslose
07. Arbeitslose
08. Arbeitslose Frauen
09. Arbeitslose Männer
10. Arbeitslose mit Anforderungsniveau Experte
11. Arbeitslose mit Anforderungsniveau Fachkraft
12. Arbeitslose mit Anforderungsniveau Helfer
13. Arbeitslose mit Anforderungsniveau Spezialist
14. Arbeitslose ohne Ausbildung
15. Arbeitslosigkeit
16. Arbeitslosigkeit - Allgemein
17. Arbeitslosigkeit - Altersgruppen
18. Arbeitslosigkeit - Struktur
19. Ausländische Arbeitslose
20. Ausländische männliche Arbeitslose
21. Ausländische weibliche Arbeitslose
22. Jüngere Arbeitslose
23. Langzeitarbeitslose
24. Männliche Langzeitarbeitslose
25. Männliche jüngere Arbeitslose
26. Männliche ältere Arbeitslose
27. Weibliche Langzeitarbeitslose
28. Weibliche jüngere Arbeitslo

In [4]:
# Cell 4 — save full indicator matches to a text file

from pathlib import Path

OUT = Path("../output")
OUT.mkdir(exist_ok=True)

match_file = OUT / "indicator_matches.txt"

keywords = ["Arbeitslos", "SGB", "65", "Bevölkerungsentwicklung"]

with open(match_file, "w", encoding="utf-8") as f:
    for keyword in keywords:
        matches = [name for name in indicator_names if keyword.lower() in name.lower()]
        f.write(f"\n--- {len(matches)} indicators containing '{keyword}' ---\n")
        for i, name in enumerate(matches, start=1):
            f.write(f"{i:02d}. {name}\n")

print("Saved full matches to:", match_file)

Saved full matches to: ..\output\indicator_matches.txt


In [5]:
# Cell 5 — inspect selected candidate indicators

CANDIDATES = [
    "Arbeitslosigkeit",
    "Arbeitslosigkeit - Allgemein",
    "Arbeitslose",
    "ALG II-Leistungen an SGBII",
    "Leistungen für Unterkunft an SGBII",
    "Einwohner 65 Jahre und älter",
    "Bevölkerungsentwicklung",
    "Bevölkerungsentwicklung (5 Jahre)",
    "Bevölkerungsentwicklung (10 Jahre)",
]

candidate_rows = []

for chunk in pd.read_csv(
    inkar_csv,
    sep=";",
    decimal=",",
    encoding="utf-8",
    dtype={"Kennziffer": str},
    chunksize=500_000
):
    part = chunk[
        (chunk["Raumbezug"] == "Gemeinden")
        & (chunk["Indikator"].isin(CANDIDATES))
        & (chunk["Kennziffer"].astype(str).str.startswith("05"))
    ].copy()
    
    if not part.empty:
        candidate_rows.append(part)

candidate_df = pd.concat(candidate_rows, ignore_index=True)

print("Rows found:", len(candidate_df))
print("Indicators found:")
print(candidate_df["Indikator"].value_counts())

print("\nYears by indicator:")
for ind in CANDIDATES:
    years = sorted(candidate_df.loc[candidate_df["Indikator"] == ind, "Zeitbezug"].dropna().unique())
    if years:
        print(ind, ":", years)

candidate_df[
    candidate_df["Name"].str.contains("Dortmund|Bochum|Essen", case=False, na=False)
][["Indikator", "Kennziffer", "Name", "Zeitbezug", "Wert"]].head(80)



Rows found: 44352
Indicators found:
Indikator
Arbeitslose                           10296
Bevölkerungsentwicklung (5 Jahre)      9504
Einwohner 65 Jahre und älter           9108
Bevölkerungsentwicklung (10 Jahre)     7524
Arbeitslosigkeit                       3168
Bevölkerungsentwicklung                2772
ALG II-Leistungen an SGBII              792
Leistungen für Unterkunft an SGBII      792
Arbeitslosigkeit - Allgemein            396
Name: count, dtype: int64

Years by indicator:
Arbeitslosigkeit : [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Arbeitslosigkeit - Allgemein : [np.int64(2023)]
Arbeitslose : [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(201

,Indikator,Kennziffer,Name,Zeitbezug,Wert
52,Arbeitslose,05113000,"Essen, Stadt",1998,33662.0
53,Arbeitslose,05113000,"Essen, Stadt",1999,32602.0
54,Arbeitslose,05113000,"Essen, Stadt",2000,31404.0
55,Arbeitslose,05113000,"Essen, Stadt",2001,30592.0
56,Arbeitslose,05113000,"Essen, Stadt",2002,32131.0
...,...,...,...,...,...
8161,Arbeitslose,05911000,"Bochum, Stadt",2021,17821.0
8162,Arbeitslose,05911000,"Bochum, Stadt",2022,16733.0
8163,Arbeitslose,05911000,"Bochum, Stadt",2023,16961.0
8164,Arbeitslose,05913000,"Dortmund, Stadt",1998,40955.0


In [6]:
# Cell 6 — inspect INKAR indicator overview / metadata

overview_xlsx = RAW / "inkar_2025" / "Indikatorenübersicht (INKAR 2025).xlsx"

print("File exists:", overview_xlsx.exists())

# First inspect sheet names
xls = pd.ExcelFile(overview_xlsx)
print("Sheets:")
print(xls.sheet_names)

File exists: True
Sheets:
['Nutzungshinweise', 'Raumbeobachtung DE', 'ZOM', 'SDG', 'Raumbeobachtung EU']


c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Raumbeobachtung DE'!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


In [7]:
# Cell 7 — read German indicator metadata sheet

meta_de = pd.read_excel(
    overview_xlsx,
    sheet_name="Raumbeobachtung DE"
)

print("Shape:", meta_de.shape)
print("Columns:")
print(list(meta_de.columns))

meta_de.head(10)

Shape: (458, 9)
Columns:
['INKAR 2025 – Indikatorenübersicht:   Raumbeobachtung Deutschland', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']


,INKAR 2025 – Indikatorenübersicht: Raumbeobachtung Deutschland,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,Kurzname,Name,Algorithmus,M_ID,Kürzel,Anmerkungen,Statistische Grundlagen,Gemeinden,Kreise
1,Absolutzahlen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bodenfläche gesamt qkm,Katasterfläche in km²,NaN,1,TN23-kataster_qkm,Als Katasterfläche bezeichnet man den vermessu...,Laufende Raumbeobachtung des BBSR; Flächenerhe...,2016-2023,2016-2023
3,Bevölkerung gesamt,Zahl der Einwohner insgesamt,NaN,2,xbev,Es handelt sich um die Zahl der Einwohner zum ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
4,Bevölkerung männlich,Zahl der männlichen Einwohner,NaN,3,xbevm,Es handelt sich um die Zahl der männlichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
5,Bevölkerung weiblich,Zahl der weiblichen Einwohner,NaN,4,xbevf,Es handelt sich um die Zahl der weiblichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
6,Bevölkerung (mit BBSR-Zensuskorrekturen),Zensuskorrigierte Zahl der Einwohner insgesamt,NaN,5,bev_korr,Zwischen der Zahl der Bevölkerung am 31.12. ei...,Laufende Raumbeobachtung des BBSR,1995-2023,1995-2023
7,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Zahl der Einwohner im Alter von 15 bis unter 6...,NaN,6,ewf_1565_ges,Es handelt sich um die Zahl der Einwohner und ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,2001-2023,1995-2023
8,Erwerbstätige,Zahl der Erwerbstätigen in 1000 Personen,NaN,7,et1000,Es handelt sich um die Zahl der Erwerbstätigen...,Laufende Raumbeobachtung des BBSR; Arbeitskrei...,NaN,2000-2022
9,Sozialversicherungspflichtig Beschäftigte am A...,Zahl der sozialversicherungspflichtig Beschäft...,NaN,8,sva,Es handelt sich um die Zahl der sozialversiche...,Laufende Raumbeobachtung des BBSR; Beschäftigt...,1997-2023,1997-2023


In [8]:
# Cell 8 — read metadata again with the correct header row

meta_de = pd.read_excel(
    overview_xlsx,
    sheet_name="Raumbeobachtung DE",
    header=1
)

print("Shape:", meta_de.shape)
print("Columns:")
print(list(meta_de.columns))

meta_de.head(10)

Shape: (457, 9)
Columns:
['Kurzname', 'Name', 'Algorithmus', 'M_ID', 'Kürzel', 'Anmerkungen', 'Statistische Grundlagen', 'Gemeinden', 'Kreise']


c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Raumbeobachtung DE'!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


,Kurzname,Name,Algorithmus,M_ID,Kürzel,Anmerkungen,Statistische Grundlagen,Gemeinden,Kreise
0,Absolutzahlen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bodenfläche gesamt qkm,Katasterfläche in km²,NaN,1.0,TN23-kataster_qkm,Als Katasterfläche bezeichnet man den vermessu...,Laufende Raumbeobachtung des BBSR; Flächenerhe...,2016-2023,2016-2023
2,Bevölkerung gesamt,Zahl der Einwohner insgesamt,NaN,2.0,xbev,Es handelt sich um die Zahl der Einwohner zum ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
3,Bevölkerung männlich,Zahl der männlichen Einwohner,NaN,3.0,xbevm,Es handelt sich um die Zahl der männlichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
4,Bevölkerung weiblich,Zahl der weiblichen Einwohner,NaN,4.0,xbevf,Es handelt sich um die Zahl der weiblichen Ein...,Laufende Raumbeobachtung des BBSR; Fortschreib...,1995-2023,1995-2023
5,Bevölkerung (mit BBSR-Zensuskorrekturen),Zensuskorrigierte Zahl der Einwohner insgesamt,NaN,5.0,bev_korr,Zwischen der Zahl der Bevölkerung am 31.12. ei...,Laufende Raumbeobachtung des BBSR,1995-2023,1995-2023
6,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Zahl der Einwohner im Alter von 15 bis unter 6...,NaN,6.0,ewf_1565_ges,Es handelt sich um die Zahl der Einwohner und ...,Laufende Raumbeobachtung des BBSR; Fortschreib...,2001-2023,1995-2023
7,Erwerbstätige,Zahl der Erwerbstätigen in 1000 Personen,NaN,7.0,et1000,Es handelt sich um die Zahl der Erwerbstätigen...,Laufende Raumbeobachtung des BBSR; Arbeitskrei...,NaN,2000-2022
8,Sozialversicherungspflichtig Beschäftigte am A...,Zahl der sozialversicherungspflichtig Beschäft...,NaN,8.0,sva,Es handelt sich um die Zahl der sozialversiche...,Laufende Raumbeobachtung des BBSR; Beschäftigt...,1997-2023,1997-2023
9,Sozialversicherungspflichtig Beschäftigte am W...,Zahl der sozialversicherungspflichtig Beschäft...,NaN,9.0,svw,Es handelt sich um die Zahl der sozialversiche...,Laufende Raumbeobachtung des BBSR; Beschäftigt...,1997-2023,1997-2023


In [9]:
# Cell 9 — inspect metadata for candidate indicators

CANDIDATES = [
    "Arbeitslosigkeit",
    "Arbeitslosigkeit - Allgemein",
    "Arbeitslose",
    "ALG II-Leistungen an SGBII",
    "Leistungen für Unterkunft an SGBII",
    "Einwohner 65 Jahre und älter",
    "Bevölkerungsentwicklung",
    "Bevölkerungsentwicklung (5 Jahre)",
    "Bevölkerungsentwicklung (10 Jahre)",
]

meta_candidates = meta_de[
    meta_de["Kurzname"].isin(CANDIDATES)
].copy()

meta_candidates[
    ["Kurzname", "Name", "Algorithmus", "Kürzel", "Anmerkungen", "Gemeinden"]
]

,Kurzname,Name,Algorithmus,Kürzel,Anmerkungen,Gemeinden
10,Arbeitslose,Zahl der Arbeitslosen insgesamt,NaN,alo,Es handelt sich um die Zahl der Arbeitslosen i...,1998-2023
12,Arbeitslosigkeit,NaN,NaN,NaN,NaN,NaN
13,Arbeitslosigkeit - Allgemein,NaN,NaN,NaN,NaN,NaN
143,Einwohner 65 Jahre und älter,Anteil der Einwohner 65 Jahre und älter an den...,E 65 Jahre und älter <Zeitpunkt> / E <Zeitpunk...,a_bev65um,Die Altersgruppe umfasst die i.d.R. nicht mehr...,2001-2023
171,Bevölkerungsentwicklung (10 Jahre),Entwicklung der Zahl der Einwohner über die le...,(E <aktueller Zeitpunkt> - E <Ausgangszeitpunk...,e10_bev,Berechnet ist die Veränderung der Bevölkerung ...,NaN
172,Bevölkerungsentwicklung (5 Jahre),Entwicklung der Zahl der Einwohner über die le...,(E <aktueller Zeitpunkt> - E <Ausgangszeitpunk...,e5_bev,Berechnet ist die Veränderung der Bevölkerung ...,NaN
386,ALG II-Leistungen an SGBII,Anteil Arbeitslosengeld II an SGBII-Leistungen...,Arbeitslosengeld II-Leistungen nach SGB II <Ze...,a_ALGII_SGBII,Die Grundsicherung für Arbeitsuchende nach dem...,NaN
388,Leistungen für Unterkunft an SGBII,Anteil Leistungen für Unterkunft an SGBII-Leis...,Leistungen für Unterkunft nach SGB II <Zeitpun...,a_Unterkunft_SGBII,Die Grundsicherung für Arbeitsuchende nach dem...,NaN


In [10]:
# Cell 10 — search metadata for better social and unemployment indicators

search_terms = [
    "Quote",
    "Arbeitslosenquote",
    "Arbeitslos",
    "SGB",
    "Mindestsicherung",
    "Transfer",
    "Empfänger",
    "Sozial",
    "Bedarf",
    "Grundsicherung"
]

for term in search_terms:
    matches = meta_de[
        meta_de["Kurzname"].astype(str).str.contains(term, case=False, na=False)
        | meta_de["Name"].astype(str).str.contains(term, case=False, na=False)
        | meta_de["Anmerkungen"].astype(str).str.contains(term, case=False, na=False)
    ].copy()
    
    print(f"\n--- {len(matches)} metadata rows containing '{term}' ---")
    for _, row in matches[["Kurzname", "Name", "Kürzel", "Gemeinden"]].head(30).iterrows():
        print(f"- {row['Kurzname']} | {row['Name']} | {row['Kürzel']} | Gemeinden: {row['Gemeinden']}")


--- 37 metadata rows containing 'Quote' ---
- Arbeitslosenquote | Anteil der Arbeitslosen an den zivilen Erwerbspersonen in % | q_alo | Gemeinden: nan
- Arbeitslosenquote Frauen | Anteil der arbeitslosen Frauen an den weiblichen zivilen Erwerbspersonen in % | q_alo_f | Gemeinden: nan
- Arbeitslosenquote Männer | Anteil der arbeitslosen Männer an den männlichen zivilen Erwerbspersonen in % | q_alo_m | Gemeinden: nan
- Arbeitslosenquote Ältere | Anteil der Arbeitslosen 55 Jahre und älter an den zivilen Erwerbspersonen 55 Jahre und älter in % | q_alo_ü55 | Gemeinden: nan
- Arbeitslosenquote Jüngere | Anteil der Arbeitslosen unter 25 Jahren an den zivilen Erwerbspersonen unter 25 Jahre in % | q_alo_u25 | Gemeinden: nan
- Anteil jüngere Arbeitslose | Anteil der Arbeitslosen unter 25 Jahren an den Arbeitslosen in % | a_alo_u25 | Gemeinden: 1998-2023
- Ältere Arbeitslose | Anteil der Arbeitslosen 55 Jahre und älter an Einwohnern von 55 bis unter 65 Jahren in % | q_alo_ü55_einw | Gemeinden: 2

In [11]:
# Cell 11 — inspect final candidate indicators in actual INKAR data

FINAL_CANDIDATES = [
    "Bevölkerungsentwicklung (5 Jahre)",
    "Einwohner 65 Jahre und älter",
    "Langzeitarbeitslose",
    "Beschäftigtenquote",
]

final_rows = []

for chunk in pd.read_csv(
    inkar_csv,
    sep=";",
    decimal=",",
    encoding="utf-8",
    dtype={"Kennziffer": str},
    chunksize=500_000
):
    part = chunk[
        (chunk["Raumbezug"] == "Gemeinden")
        & (chunk["Indikator"].isin(FINAL_CANDIDATES))
        & (chunk["Kennziffer"].astype(str).str.startswith("05"))
    ].copy()
    
    if not part.empty:
        final_rows.append(part)

final_df = pd.concat(final_rows, ignore_index=True)

print("Rows found:", len(final_df))
print("\nIndicators found:")
print(final_df["Indikator"].value_counts())

print("\nYears by indicator:")
for ind in FINAL_CANDIDATES:
    years = sorted(final_df.loc[final_df["Indikator"] == ind, "Zeitbezug"].dropna().unique())
    print(ind, ":", years)

final_df[
    final_df["Name"].str.contains("Dortmund|Bochum|Essen", case=False, na=False)
][["Indikator", "Kennziffer", "Name", "Zeitbezug", "Wert"]].sort_values(
    ["Name", "Indikator", "Zeitbezug"]
).tail(80)

Rows found: 38010

Indicators found:
Indikator
Langzeitarbeitslose                  10296
Bevölkerungsentwicklung (5 Jahre)     9504
Einwohner 65 Jahre und älter          9108
Beschäftigtenquote                    9102
Name: count, dtype: int64

Years by indicator:
Bevölkerungsentwicklung (5 Jahre) : [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Einwohner 65 Jahre und älter : [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(

,Indikator,Kennziffer,Name,Zeitbezug,Wert
35245,Beschäftigtenquote,05762040,"Willebadessen, Stadt",2017,58.21
35246,Beschäftigtenquote,05762040,"Willebadessen, Stadt",2018,60.50
35247,Beschäftigtenquote,05762040,"Willebadessen, Stadt",2019,61.34
35248,Beschäftigtenquote,05762040,"Willebadessen, Stadt",2020,61.36
35249,Beschäftigtenquote,05762040,"Willebadessen, Stadt",2021,62.75
...,...,...,...,...,...
16675,Langzeitarbeitslose,05762040,"Willebadessen, Stadt",2019,29.61
16676,Langzeitarbeitslose,05762040,"Willebadessen, Stadt",2020,25.99
16677,Langzeitarbeitslose,05762040,"Willebadessen, Stadt",2021,35.85
16678,Langzeitarbeitslose,05762040,"Willebadessen, Stadt",2022,31.10


In [12]:
# Cell 12 — check completeness for final indicators in 2023

CHECK_YEAR = 2023

check_2023 = final_df[final_df["Zeitbezug"] == CHECK_YEAR].copy()

print("Rows in 2023:", len(check_2023))
print("\nRows per indicator in 2023:")
print(check_2023["Indikator"].value_counts())

print("\nNumber of unique Gemeinden per indicator in 2023:")
for ind in FINAL_CANDIDATES:
    n = check_2023.loc[
        check_2023["Indikator"] == ind,
        "Kennziffer"
    ].nunique()
    print(f"{ind}: {n}")

Rows in 2023: 1584

Rows per indicator in 2023:
Indikator
Bevölkerungsentwicklung (5 Jahre)    396
Langzeitarbeitslose                  396
Einwohner 65 Jahre und älter         396
Beschäftigtenquote                   396
Name: count, dtype: int64

Number of unique Gemeinden per indicator in 2023:
Bevölkerungsentwicklung (5 Jahre): 396
Einwohner 65 Jahre und älter: 396
Langzeitarbeitslose: 396
Beschäftigtenquote: 396


In [13]:
# Cell 13 — final indicator-year selection

INDICATOR_YEAR = {
    "Bevölkerungsentwicklung (5 Jahre)": 2023,
    "Einwohner 65 Jahre und älter": 2023,
    "Langzeitarbeitslose": 2023,
    "Beschäftigtenquote": 2023,
}

inkar_filtered = final_df[
    final_df.apply(
        lambda row: INDICATOR_YEAR.get(row["Indikator"]) == row["Zeitbezug"],
        axis=1
    )
].copy()

print("Filtered rows:", len(inkar_filtered))
print("\nRows per indicator:")
print(inkar_filtered["Indikator"].value_counts())

inkar_filtered[["Kennziffer", "Name", "Indikator", "Zeitbezug", "Wert"]].head(12)

Filtered rows: 1584

Rows per indicator:
Indikator
Bevölkerungsentwicklung (5 Jahre)    396
Langzeitarbeitslose                  396
Einwohner 65 Jahre und älter         396
Beschäftigtenquote                   396
Name: count, dtype: int64


,Kennziffer,Name,Indikator,Zeitbezug,Wert
23,05111000,"Düsseldorf, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,-0.48
47,05112000,"Duisburg, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,0.92
71,05113000,"Essen, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,-1.55
95,05114000,"Krefeld, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,1.49
119,05116000,"Mönchengladbach, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,2.38
143,05117000,"Mülheim an der Ruhr, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,1.17
167,05119000,"Oberhausen, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,1.27
191,05120000,"Remscheid, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,2.76
215,05122000,"Solingen, Klingenstadt",Bevölkerungsentwicklung (5 Jahre),2023,4.22
239,05124000,"Wuppertal, Stadt",Bevölkerungsentwicklung (5 Jahre),2023,1.19


In [14]:
# Cell 14 — pivot INKAR from long to wide format

inkar_wide = inkar_filtered.pivot_table(
    index=["Kennziffer", "Name"],
    columns="Indikator",
    values="Wert",
    aggfunc="first"
).reset_index()

# Rename Kennziffer to AGS for joining with BKG geometry later
inkar_wide = inkar_wide.rename(columns={"Kennziffer": "AGS"})

# Preserve leading zeros and force 8-character municipality key
inkar_wide["AGS"] = inkar_wide["AGS"].astype(str).str.zfill(8)

print("Wide shape:", inkar_wide.shape)
print("Columns:")
print(list(inkar_wide.columns))

inkar_wide.head(10)

Wide shape: (396, 6)
Columns:
['AGS', 'Name', 'Beschäftigtenquote', 'Bevölkerungsentwicklung (5 Jahre)', 'Einwohner 65 Jahre und älter', 'Langzeitarbeitslose']


Indikator,AGS,Name,Beschäftigtenquote,Bevölkerungsentwicklung (5 Jahre),Einwohner 65 Jahre und älter,Langzeitarbeitslose
0,05111000,"Düsseldorf, Stadt",63.84,-0.48,18.98,34.94
1,05112000,"Duisburg, Stadt",56.24,0.92,20.25,44.18
2,05113000,"Essen, Stadt",58.96,-1.55,21.84,42.69
3,05114000,"Krefeld, Stadt",59.35,1.49,22.14,46.39
4,05116000,"Mönchengladbach, Stadt",58.93,2.38,21.41,44.22
5,05117000,"Mülheim an der Ruhr, Stadt",59.37,1.17,23.77,52.67
6,05119000,"Oberhausen, Stadt",57.51,1.27,22.57,49.14
7,05120000,"Remscheid, Stadt",60.77,2.76,22.04,33.77
8,05122000,"Solingen, Klingenstadt",60.86,4.22,21.88,46.63
9,05124000,"Wuppertal, Stadt",58.17,1.19,20.89,41.12


In [15]:
# Cell 15 — clean column names

inkar_wide.columns.name = None

inkar_wide = inkar_wide.rename(columns={
    "Beschäftigtenquote": "employment_rate_pct",
    "Bevölkerungsentwicklung (5 Jahre)": "population_change_5y_pct",
    "Einwohner 65 Jahre und älter": "share_65plus_pct",
    "Langzeitarbeitslose": "longterm_unemployed_share_pct",
})

print("Shape:", inkar_wide.shape)
print("Columns:")
print(list(inkar_wide.columns))

inkar_wide.head(10)

Shape: (396, 6)
Columns:
['AGS', 'Name', 'employment_rate_pct', 'population_change_5y_pct', 'share_65plus_pct', 'longterm_unemployed_share_pct']


,AGS,Name,employment_rate_pct,population_change_5y_pct,share_65plus_pct,longterm_unemployed_share_pct
0,05111000,"Düsseldorf, Stadt",63.84,-0.48,18.98,34.94
1,05112000,"Duisburg, Stadt",56.24,0.92,20.25,44.18
2,05113000,"Essen, Stadt",58.96,-1.55,21.84,42.69
3,05114000,"Krefeld, Stadt",59.35,1.49,22.14,46.39
4,05116000,"Mönchengladbach, Stadt",58.93,2.38,21.41,44.22
5,05117000,"Mülheim an der Ruhr, Stadt",59.37,1.17,23.77,52.67
6,05119000,"Oberhausen, Stadt",57.51,1.27,22.57,49.14
7,05120000,"Remscheid, Stadt",60.77,2.76,22.04,33.77
8,05122000,"Solingen, Klingenstadt",60.86,4.22,21.88,46.63
9,05124000,"Wuppertal, Stadt",58.17,1.19,20.89,41.12


In [16]:
# Cell 16 — save cleaned INKAR wide table

PROC.mkdir(exist_ok=True)

inkar_out = PROC / "inkar_nrw_wide_2023.csv"

inkar_wide.to_csv(inkar_out, index=False, encoding="utf-8")

print("Saved:", inkar_out)
print("Rows:", len(inkar_wide))
print("Columns:", list(inkar_wide.columns))

Saved: ..\data\processed\inkar_nrw_wide_2023.csv
Rows: 396
Columns: ['AGS', 'Name', 'employment_rate_pct', 'population_change_5y_pct', 'share_65plus_pct', 'longterm_unemployed_share_pct']


In [17]:
# Cell 17 — load BKG Gemeinde geometry and filter NRW

import geopandas as gpd
import zipfile

bkg_zip = RAW / "vg250_01-01.utm32s.shape.ebenen.zip"

with zipfile.ZipFile(bkg_zip, "r") as z:
    shp_files = [name for name in z.namelist() if name.endswith("VG250_GEM.shp")]

print("Matching shapefiles:")
print(shp_files)

gem_path = f"zip://{bkg_zip}!{shp_files[0]}"

gem = gpd.read_file(gem_path)

print("Full BKG Gemeinden shape:", gem.shape)
print("CRS:", gem.crs)
print("Columns:")
print(list(gem.columns))

gem.head(3)

Matching shapefiles:
['vg250_01-01.utm32s.shape.ebenen/vg250_ebenen_0101/VG250_GEM.shp']
Full BKG Gemeinden shape: (11103, 27)
CRS: EPSG:25832
Columns:
['OBJID', 'BEGINN', 'ADE', 'GF', 'BSG', 'LKZ', 'ARS', 'AGS', 'SDV_ARS', 'GEN', 'BEZ', 'IBZ', 'BEM', 'NBD', 'SN_L', 'SN_R', 'SN_K', 'SN_V1', 'SN_V2', 'SN_G', 'FK_S3', 'NUTS', 'ARS_0', 'AGS_0', 'WSK', 'DLM_ID', 'geometry']


,OBJID,BEGINN,ADE,GF,BSG,LKZ,ARS,AGS,SDV_ARS,GEN,...,SN_V1,SN_V2,SN_G,FK_S3,NUTS,ARS_0,AGS_0,WSK,DLM_ID,geometry
0,DEBKGVG200000008,2022-12-20,6,4,1,SH,010010000000,01001000,010010000000,Flensburg,...,00,00,000,R,DEF01,010010000000,01001000,2008-01-01,DEBKGDL20000E5MA,"POLYGON ((527173.204 6075202.115, 527199.066 6..."
1,DEBKGVG200000009,2025-01-04,6,4,1,SH,010020000000,01002000,010020000000,Kiel,...,00,00,000,R,DEF02,010020000000,01002000,2006-01-01,DEBKGDL20000004J,"POLYGON ((575797.258 6032310.691, 575841.569 6..."
2,DEBKGVG20000000A,2025-01-04,6,4,1,SH,010030000000,01003000,010030000000,Lübeck,...,00,00,000,R,DEF03,010030000000,01003000,2006-02-01,DEBKGDL20000DYMA,"POLYGON ((623482.947 5981779.206, 623460.091 5..."


In [18]:
# Cell 18 — filter NRW Gemeinden and keep only needed columns

gem_nrw = gem[gem["SN_L"] == "05"].copy()

# Keep only the columns needed for joining, labeling, and mapping
gem_nrw = gem_nrw[["AGS", "GEN", "BEZ", "SN_L", "geometry"]].copy()

# Make sure AGS is text and keeps leading zeros
gem_nrw["AGS"] = gem_nrw["AGS"].astype(str).str.zfill(8)

print("NRW Gemeinden shape:", gem_nrw.shape)
print("CRS:", gem_nrw.crs)
print("Columns:")
print(list(gem_nrw.columns))

gem_nrw.head(10)

NRW Gemeinden shape: (396, 5)
CRS: EPSG:25832
Columns:
['AGS', 'GEN', 'BEZ', 'SN_L', 'geometry']


,AGS,GEN,BEZ,SN_L,geometry
2073,05111000,Düsseldorf,Stadt,05,"POLYGON ((347157.737 5690900.882, 347138.594 5..."
2074,05112000,Duisburg,Stadt,05,"POLYGON ((346303.285 5711452.756, 346278.773 5..."
2075,05113000,Essen,Stadt,05,"POLYGON ((361988.204 5710945.715, 362206.664 5..."
2076,05114000,Krefeld,Stadt,05,"POLYGON ((331703.174 5696424.089, 331838.571 5..."
2077,05116000,Mönchengladbach,Stadt,05,"POLYGON ((326986.397 5677844.615, 326860.383 5..."
2078,05117000,Mülheim an der Ruhr,Stadt,05,"POLYGON ((354078.488 5692771.027, 353890.72 56..."
2079,05119000,Oberhausen,Stadt,05,"POLYGON ((349863.592 5716511.905, 349927.816 5..."
2080,05120000,Remscheid,Stadt,05,"POLYGON ((380949.914 5674045.756, 380951.404 5..."
2081,05122000,Solingen,Stadt,05,"POLYGON ((369682.942 5669978.13, 369746.776 56..."
2082,05124000,Wuppertal,Stadt,05,"POLYGON ((378873.921 5685099.808, 378687.641 5..."


In [19]:
# Cell 19 — check AGS match between INKAR table and BKG geometry

inkar_ags = set(inkar_wide["AGS"])
bkg_ags = set(gem_nrw["AGS"])

print("INKAR AGS count:", len(inkar_ags))
print("BKG AGS count:", len(bkg_ags))
print("Common AGS count:", len(inkar_ags & bkg_ags))

print("\nIn INKAR but not in BKG:")
print(sorted(inkar_ags - bkg_ags)[:20])

print("\nIn BKG but not in INKAR:")
print(sorted(bkg_ags - inkar_ags)[:20])

INKAR AGS count: 396
BKG AGS count: 396
Common AGS count: 396

In INKAR but not in BKG:
[]

In BKG but not in INKAR:
[]


In [20]:
# Cell 20 — join INKAR indicators to BKG NRW geometry

nrw_indicators = gem_nrw.merge(
    inkar_wide,
    on="AGS",
    how="left"
)

print("Joined shape:", nrw_indicators.shape)
print("CRS:", nrw_indicators.crs)
print("Missing values after join:")
print(nrw_indicators[
    [
        "employment_rate_pct",
        "population_change_5y_pct",
        "share_65plus_pct",
        "longterm_unemployed_share_pct"
    ]
].isna().sum())

nrw_indicators.head(10)

Joined shape: (396, 10)
CRS: EPSG:25832
Missing values after join:
employment_rate_pct              0
population_change_5y_pct         0
share_65plus_pct                 0
longterm_unemployed_share_pct    0
dtype: int64


,AGS,GEN,BEZ,SN_L,geometry,Name,employment_rate_pct,population_change_5y_pct,share_65plus_pct,longterm_unemployed_share_pct
0,05111000,Düsseldorf,Stadt,05,"POLYGON ((347157.737 5690900.882, 347138.594 5...","Düsseldorf, Stadt",63.84,-0.48,18.98,34.94
1,05112000,Duisburg,Stadt,05,"POLYGON ((346303.285 5711452.756, 346278.773 5...","Duisburg, Stadt",56.24,0.92,20.25,44.18
2,05113000,Essen,Stadt,05,"POLYGON ((361988.204 5710945.715, 362206.664 5...","Essen, Stadt",58.96,-1.55,21.84,42.69
3,05114000,Krefeld,Stadt,05,"POLYGON ((331703.174 5696424.089, 331838.571 5...","Krefeld, Stadt",59.35,1.49,22.14,46.39
4,05116000,Mönchengladbach,Stadt,05,"POLYGON ((326986.397 5677844.615, 326860.383 5...","Mönchengladbach, Stadt",58.93,2.38,21.41,44.22
5,05117000,Mülheim an der Ruhr,Stadt,05,"POLYGON ((354078.488 5692771.027, 353890.72 56...","Mülheim an der Ruhr, Stadt",59.37,1.17,23.77,52.67
6,05119000,Oberhausen,Stadt,05,"POLYGON ((349863.592 5716511.905, 349927.816 5...","Oberhausen, Stadt",57.51,1.27,22.57,49.14
7,05120000,Remscheid,Stadt,05,"POLYGON ((380949.914 5674045.756, 380951.404 5...","Remscheid, Stadt",60.77,2.76,22.04,33.77
8,05122000,Solingen,Stadt,05,"POLYGON ((369682.942 5669978.13, 369746.776 56...","Solingen, Klingenstadt",60.86,4.22,21.88,46.63
9,05124000,Wuppertal,Stadt,05,"POLYGON ((378873.921 5685099.808, 378687.641 5...","Wuppertal, Stadt",58.17,1.19,20.89,41.12


In [21]:
# Cell 21 — save joined NRW indicator GeoPackage

gpkg_out = PROC / "nrw_indicators_2023.gpkg"

nrw_indicators.to_file(
    gpkg_out,
    layer="nrw_indicators_2023",
    driver="GPKG"
)

print("Saved:", gpkg_out)
print("Rows:", len(nrw_indicators))
print("CRS:", nrw_indicators.crs)

Saved: ..\data\processed\nrw_indicators_2023.gpkg
Rows: 396
CRS: EPSG:25832


In [22]:
# Cell 22 — verify saved GeoPackage

test_gpkg = gpd.read_file(gpkg_out, layer="nrw_indicators_2023")

print("Loaded shape:", test_gpkg.shape)
print("CRS:", test_gpkg.crs)
print("Columns:")
print(list(test_gpkg.columns))

test_gpkg.head(3)


Loaded shape: (396, 10)
CRS: EPSG:25832
Columns:
['AGS', 'GEN', 'BEZ', 'SN_L', 'Name', 'employment_rate_pct', 'population_change_5y_pct', 'share_65plus_pct', 'longterm_unemployed_share_pct', 'geometry']


,AGS,GEN,BEZ,SN_L,Name,employment_rate_pct,population_change_5y_pct,share_65plus_pct,longterm_unemployed_share_pct,geometry
0,05111000,Düsseldorf,Stadt,05,"Düsseldorf, Stadt",63.84,-0.48,18.98,34.94,"MULTIPOLYGON (((347157.737 5690900.882, 347138..."
1,05112000,Duisburg,Stadt,05,"Duisburg, Stadt",56.24,0.92,20.25,44.18,"MULTIPOLYGON (((346303.285 5711452.756, 346278..."
2,05113000,Essen,Stadt,05,"Essen, Stadt",58.96,-1.55,21.84,42.69,"MULTIPOLYGON (((361988.204 5710945.715, 362206..."
